In [4]:
%pip install proc-tap psutil

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import time
import psutil
try:
    from proctap import ProcessAudioCapture, StreamConfig
except ImportError:
    try:
        from proc_tap import ProcessAudioCapture, StreamConfig
    except ImportError:
        try:
            from proctap import ProcessAudioCapture
        except ImportError as exc:
            raise ImportError(
                "Missing proc-tap dependency. Install it in the notebook with:\n"
                "  %pip install proc-tap psutil\n"
                "Then restart the kernel and run the cell again."
            ) from exc

        class StreamConfig:
            def __init__(self, sample_rate=16000, channels=1):
                self.sample_rate = sample_rate
                self.channels = channels

def find_pid_by_window_title(search_string):
    """
    Scans active processes to find a match for window titles 
    (e.g., "YouTube", "Zoom Meeting").
    """
    search_string = search_string.lower()
    print(f"Scanning for processes matching: '{search_string}'...")
    
    # We prioritize processes known to output audio/video
    for proc in psutil.process_iter(['pid', 'name']):
        try:
            # Check the process name itself first
            if search_string in proc.info['name'].lower():
                return proc.info['pid']
                
            # If the process has an open window, check the window text
            # (Works reliably for Zoom desktop client and native apps)
            cmdline = proc.cmdline()
            for arg in cmdline:
                if search_string in arg.lower():
                    return proc.info['pid']
        except (psutil.NoSuchProcess, psutil.AccessDenied, psutil.ZombieProcess):
            continue
            
    return None

def on_audio_received(pcm_bytes, frame_count):
    """
    This callback triggers in real-time as the app emits audio.
    pcm_bytes contains raw 16kHz float32 PCM audio data.
    """
    # 💡 THIS IS WHERE YOU FEED THE DATA TO YOUR AI TRANSCRIPTION PIPELINE
    # Example: whisper_model.insert_audio_chunk(pcm_bytes)
    print(f"[Captured Chunks] Received {len(pcm_bytes)} bytes of app audio.")

if __name__ == "__main__":
    # Change this to "Zoom" or "YouTube" depending on what you want to record
    TARGET_APP = "chrome" 
    
    # 1. Automate finding the process ID
    target_pid = find_pid_by_window_title(TARGET_APP)
    
    # Special fallback handler for Web Browsers (YouTube running in a tab)
    if not target_pid and TARGET_APP.lower() == "youtube":
        print("💡 Tip: YouTube runs inside web browsers. Trying to hook Chrome/Edge...")
        target_pid = find_pid_by_window_title("chrome") or find_pid_by_window_title("msedge")

    if not target_pid:
        print(f"❌ Could not find an active process for '{TARGET_APP}'. Make sure it is open and playing sound!")
        exit()

    print(f"✅ Found target process! Target PID: {target_pid}")

    # 2. Configure audio stream optimized for AI Transcription (16kHz, Mono)
    config = StreamConfig(sample_rate=16000, channels=1)

    # 3. Initialize loopback interception on the targeted application
    try:
        print(f"🎧 Capturing audio exclusively from PID {target_pid}. Press Ctrl+C to stop.")
        with ProcessAudioCapture(target_pid, config=config, on_data=on_audio_received):
            while True:
                time.sleep(1)  # Keeps main thread alive while callback runs
    except KeyboardInterrupt:
        print("\n⏹️ Capture stopped by user.")
    except Exception as e:
        print(f"❌ Error binding to process: {e}")


Scanning for processes matching: 'chrome'...
✅ Found target process! Target PID: 888
🎧 Capturing audio exclusively from PID 888. Press Ctrl+C to stop.
❌ Error binding to process: ProcessAudioCapture.__init__() got an unexpected keyword argument 'config'
